In [1]:
# 1. pip install
%pip install pandas numpy requests jupyter python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 🚧 서울시 도로굴착 공사현황 데이터 전처리
# FR-005: 음성 안내 서비스 - 데이터 전처리 단계

import pandas as pd
import numpy as np
import requests
import json
import os
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')

# 환경변수 로드
load_dotenv()


True

In [3]:

print("🚧 FR-005: 도로굴착 공사현황 데이터 전처리")
print("🎯 단순화 버전 - 핵심 정보만 활용")
print("🔐 환경변수 로드 완료")
print("=" * 70)

# =============================================================================
# 0. 카카오 API 연결 테스트
# =============================================================================

def test_kakao_api():
    """카카오 API 연결 상태 확인"""
    print("🔍 카카오 API 연결 테스트...")
    
    api_key = os.getenv("KAKAO_REST_API_KEY")
    print(f"🔑 API 키: {api_key[:10] if api_key else 'None'}...")
    
    if not api_key:
        print("❌ API 키가 설정되지 않았습니다.")
        return False
    
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key}"}
    params = {"query": "서울시 강남구"}
    
    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if data.get('documents'):
                first_result = data['documents'][0]
                lat = first_result.get('y')
                lng = first_result.get('x')
                print(f"✅ API 연결 성공!")
                print(f"   📍 테스트 결과: 위도 {lat}, 경도 {lng}")
                return True
            else:
                print("❌ API 응답 데이터 없음")
                return False
        else:
            print(f"❌ API 오류: {response.status_code}")
            print(f"   응답: {response.text[:100]}")
            return False
            
    except Exception as e:
        print(f"❌ API 테스트 실패: {str(e)}")
        return False

# API 테스트 실행
api_available = test_kakao_api()
print(f"🌐 API 사용 가능: {'Yes' if api_available else 'No (구별 중심좌표 사용)'}")
print("-" * 70)


🚧 FR-005: 도로굴착 공사현황 데이터 전처리
🎯 단순화 버전 - 핵심 정보만 활용
🔐 환경변수 로드 완료
🔍 카카오 API 연결 테스트...
🔑 API 키: 7730cd19fd...
✅ API 연결 성공!
   📍 테스트 결과: 위도 37.5173319258532, 경도 127.047377408384
🌐 API 사용 가능: Yes
----------------------------------------------------------------------


In [4]:

# =============================================================================
# 1. 데이터 로드 및 기본 정보 확인
# =============================================================================

def load_road_excavation_data(file_path="../../data/csv/서울시 도로굴착 공사 현황.csv"):
    """
    도로굴착 현황 CSV 파일 로드
    
    Args:
        file_path (str): CSV 파일 경로
    
    Returns:
        pd.DataFrame: 로드된 데이터프레임
    """
    try:
        # 여러 인코딩으로 시도
        encodings = ['cp949', 'euc-kr', 'utf-8']
        
        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, encoding=encoding)
                print(f"✅ 파일 로드 성공 (인코딩: {encoding})")
                break
            except UnicodeDecodeError:
                continue
        else:
            raise Exception("모든 인코딩 시도 실패")
            
        # 기본 정보 출력
        print(f"📊 총 데이터: {len(df):,}건")
        print(f"📋 원본 컬럼: {len(df.columns)}개")
        print(f"📅 컬럼 목록: {list(df.columns)}")
        
        return df
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        print("💡 파일 경로를 확인하거나 다음 경로들을 시도해보세요:")
        possible_paths = [
            "data/csv/서울시 도로굴착현황.csv",
            "./서울시 도로굴착현황.csv",
            "../data/csv/서울시 도로굴착현황.csv"
        ]
        for path in possible_paths:
            print(f"   - {path}")
        return None
    except Exception as e:
        print(f"❌ 파일 로드 중 오류 발생: {str(e)}")
        return None

# 데이터 로드
print("\n📂 데이터 로드 시작...")
df_raw = load_road_excavation_data()

if df_raw is not None:
    print(f"\n📋 데이터 미리보기 (첫 3행):")
    display_cols = df_raw.columns[:5]  # 첫 5개 컬럼만 표시
    print(df_raw[display_cols].head(3))
    
    # 자치구별 분포 확인
    if '구' in df_raw.columns:
        print(f"\n🏢 자치구별 분포 (상위 10개):")
        district_counts = df_raw['구'].value_counts()
        for district, count in district_counts.head(10).items():
            print(f"   - {district}: {count:,}건")
        print(f"   총 {len(district_counts)}개 자치구, 전체 {district_counts.sum():,}건")



📂 데이터 로드 시작...
✅ 파일 로드 성공 (인코딩: cp949)
📊 총 데이터: 533건
📋 원본 컬럼: 10개
📅 컬럼 목록: ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일', '신청자', '처리상태', '도로', '포장', '허가번호']

📋 데이터 미리보기 (첫 3행):
           관리코드    구    동                                                공사명  \
0  일반-210728-75  서초구  잠원동  잠원동 신반포14차조합(롯데건설)3000kW 신설_보도(대선, 송영아)<BR>(서초...   
1  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(...   
2  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(...   

                 착공일 ~ 준공일  
0  2022-06-20 ~ 2022-08-31  
1  2022-06-16 ~ 2022-08-14  
2  2022-06-16 ~ 2022-08-14  

🏢 자치구별 분포 (상위 10개):
   - 서초구: 192건
   - 강남구: 94건
   - 성북구: 37건
   - 관악구: 33건
   - 강동구: 33건
   - 강북구: 19건
   - 종로구: 19건
   - 도봉구: 18건
   - 금천구: 18건
   - 노원구: 15건
   총 23개 자치구, 전체 533건


In [5]:
# =============================================================================
# 2. 단순화된 데이터 전처리
# =============================================================================

def create_simplified_construction_data(df_raw):
    """
    핵심 정보만 추출하여 단순화된 공사 데이터 생성
    
    Args:
        df_raw (pd.DataFrame): 원본 데이터프레임
    
    Returns:
        pd.DataFrame: 단순화된 데이터프레임
    """
    print("\n🎯 단순화된 공사 데이터 생성...")
    
    # 1. 핵심 컬럼만 선택
    essential_cols = ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일']
    
    # 컬럼 존재 여부 확인
    missing_cols = [col for col in essential_cols if col not in df_raw.columns]
    if missing_cols:
        print(f"❌ 필수 컬럼 누락: {missing_cols}")
        print(f"📋 사용 가능한 컬럼: {list(df_raw.columns)}")
        return None
    
    df_simple = df_raw[essential_cols].copy()
    print(f"✅ 핵심 컬럼 추출 완료: {len(df_simple)}건")
    
    # 2. HTML 태그 제거 및 주소 추출
    print("🧹 텍스트 정제 중...")
    df_simple['공사명_정제'] = df_simple['공사명'].str.replace('<BR>', ' ', regex=False)
    df_simple['공사명_정제'] = df_simple['공사명_정제'].str.replace('<br>', ' ', regex=False)
    
    # 괄호 안 주소 추출 (다양한 패턴 시도)
    patterns = [
        r'\(([^)]*구[^)]*동[^)]*)\)',  # 구+동 패턴
        r'\(([^)]*구[^)]*)\)',        # 구만 있는 패턴
        r'\(([^)]*\d+[^)]*)\)'        # 번지수 패턴
    ]
    
    df_simple['추출주소'] = None
    for i, pattern in enumerate(patterns):
        mask = df_simple['추출주소'].isna()
        if mask.sum() > 0:
            extracted = df_simple.loc[mask, '공사명_정제'].str.extract(pattern)[0]
            df_simple.loc[mask, '추출주소'] = extracted
            print(f"   패턴 {i+1}: {extracted.notna().sum()}건 추출")
    
    extracted_count = df_simple['추출주소'].notna().sum()
    print(f"✅ 주소 추출 완료: {extracted_count:,}건 ({extracted_count/len(df_simple)*100:.1f}%)")
    
    # 3. 날짜 파싱
    print("📅 날짜 정보 파싱 중...")
    date_parts = df_simple['착공일 ~ 준공일'].str.split(' ~ ', expand=True)
    df_simple['착공일'] = pd.to_datetime(date_parts[0], errors='coerce')
    df_simple['준공일'] = pd.to_datetime(date_parts[1], errors='coerce')
    
    valid_start = df_simple['착공일'].notna().sum()
    valid_end = df_simple['준공일'].notna().sum()
    print(f"✅ 날짜 파싱 완료: 착공일 {valid_start:,}건, 준공일 {valid_end:,}건")
    
    # 날짜 범위 확인
    if valid_start > 0:
        min_date = df_simple['착공일'].min()
        max_date = df_simple['준공일'].max()
        print(f"   📅 날짜 범위: {min_date.strftime('%Y-%m-%d')} ~ {max_date.strftime('%Y-%m-%d')}")
    
    # 4. 현재 공사 상태 계산 (단순하게 3가지만)
    print("🔍 공사 상태 계산 중...")
    today = datetime.now()
    
    conditions = [
        df_simple['착공일'] > today,  # 미착공
        (df_simple['착공일'] <= today) & (df_simple['준공일'] >= today),  # 진행중
        df_simple['준공일'] < today   # 완료
    ]
    choices = ['미착공', '진행중', '완료']
    df_simple['공사상태'] = np.select(conditions, choices, default='불명')
    
    print("✅ 공사 상태 분류 완료:")
    status_counts = df_simple['공사상태'].value_counts()
    for status, count in status_counts.items():
        print(f"   - {status}: {count:,}건 ({count/len(df_simple)*100:.1f}%)")
    
    # 5. 지오코딩용 주소 생성
    print("🗺️ 지오코딩용 주소 생성 중...")
    
    # 우선순위: 추출주소 > 구+동 조합
    mask_extracted = df_simple['추출주소'].notna()
    df_simple.loc[mask_extracted, '지오코딩주소'] = '서울시 ' + df_simple.loc[mask_extracted, '추출주소']
    df_simple.loc[~mask_extracted, '지오코딩주소'] = '서울시 ' + df_simple.loc[~mask_extracted, '구'] + ' ' + df_simple.loc[~mask_extracted, '동']
    
    print(f"✅ 지오코딩 주소 생성 완료:")
    print(f"   - 추출주소 기반: {mask_extracted.sum():,}건")
    print(f"   - 구+동 기반: {(~mask_extracted).sum():,}건")
    
    # 지오코딩 주소 샘플 출력
    print(f"   📍 주소 샘플:")
    for i, addr in enumerate(df_simple['지오코딩주소'].head(3), 1):
        print(f"      {i}. {addr}")
    
    # 6. 최종 컬럼 정리
    final_cols = ['관리코드', '지오코딩주소', '착공일', '준공일', '공사상태']
    df_final = df_simple[final_cols].copy()
    
    print(f"\n📊 단순화 완료:")
    print(f"   - 최종 데이터: {len(df_final):,}건")
    print(f"   - 최종 컬럼: {list(df_final.columns)}")
    
    return df_final

# 단순화된 전처리 실행
if df_raw is not None:
    df_simplified = create_simplified_construction_data(df_raw)
    
    if df_simplified is not None:
        print(f"\n📋 단순화된 데이터 미리보기:")
        print(df_simplified.head(3))


🎯 단순화된 공사 데이터 생성...
✅ 핵심 컬럼 추출 완료: 533건
🧹 텍스트 정제 중...
   패턴 1: 533건 추출
✅ 주소 추출 완료: 533건 (100.0%)
📅 날짜 정보 파싱 중...
✅ 날짜 파싱 완료: 착공일 372건, 준공일 372건
   📅 날짜 범위: 2020-08-27 ~ 2026-07-14
🔍 공사 상태 계산 중...
✅ 공사 상태 분류 완료:
   - 완료: 338건 (63.4%)
   - 불명: 161건 (30.2%)
   - 진행중: 34건 (6.4%)
🗺️ 지오코딩용 주소 생성 중...
✅ 지오코딩 주소 생성 완료:
   - 추출주소 기반: 533건
   - 구+동 기반: 0건
   📍 주소 샘플:
      1. 서울시 서초구 잠원동   잠원동112 ~ 잠원동   잠원동112
      2. 서울시 용산구 동자동   35-41 ~ 동자동   35-41부근
      3. 서울시 용산구 동자동   19-87 ~ 동자동   19-87부근

📊 단순화 완료:
   - 최종 데이터: 533건
   - 최종 컬럼: ['관리코드', '지오코딩주소', '착공일', '준공일', '공사상태']

📋 단순화된 데이터 미리보기:
           관리코드                               지오코딩주소        착공일        준공일  \
0  일반-210728-75  서울시 서초구 잠원동   잠원동112 ~ 잠원동   잠원동112 2022-06-20 2022-08-31   
1  일반-210527-84  서울시 용산구 동자동   35-41 ~ 동자동   35-41부근 2022-06-16 2022-08-14   
2  일반-210527-84  서울시 용산구 동자동   19-87 ~ 동자동   19-87부근 2022-06-16 2022-08-14   

  공사상태  
0   완료  
1   완료  
2   완료  


In [6]:
# =============================================================================
# 3. 지오코딩 (카카오맵 API + 백업 좌표)
# =============================================================================

class SimpleKakaoGeocoder:
    """단순화된 카카오맵 지오코딩 서비스"""
    
    def __init__(self):
        self.api_key = os.getenv('KAKAO_REST_API_KEY')
        self.base_url = "https://dapi.kakao.com/v2/local/search/address.json"
        self.delay = 0.2  # API 호출 간격 (안전하게 설정)
        
        # 서울시 자치구별 중심 좌표 (API 실패 시 사용)
        self.district_coords = {
            '강남구': (37.4979, 127.0276), '강동구': (37.5301, 127.1238),
            '강북구': (37.6398, 127.0256), '강서구': (37.5509, 126.8495),
            '관악구': (37.4784, 126.9516), '광진구': (37.5385, 127.0823),
            '구로구': (37.4955, 126.8874), '금천구': (37.4563, 126.8956),
            '노원구': (37.6542, 127.0568), '도봉구': (37.6688, 127.0471),
            '동대문구': (37.5744, 127.0396), '동작구': (37.5124, 126.9393),
            '마포구': (37.5637, 126.9084), '서대문구': (37.5794, 126.9368),
            '서초구': (37.4837, 127.0324), '성동구': (37.5634, 127.0365),
            '성북구': (37.5894, 127.0167), '송파구': (37.5145, 127.1059),
            '양천구': (37.5168, 126.8664), '영등포구': (37.5264, 126.8962),
            '용산구': (37.5326, 126.9905), '은평구': (37.6028, 126.9292),
            '종로구': (37.5735, 126.9788), '중구': (37.5641, 126.9979),
            '중랑구': (37.6063, 127.0925)
        }
    
    def get_coordinates(self, address):
        """단일 주소를 좌표로 변환"""
        if not self.api_key:
            return None
            
        try:
            headers = {"Authorization": f"KakaoAK {self.api_key}"}
            params = {"query": address}
            
            time.sleep(self.delay)
            response = requests.get(self.base_url, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                if data['documents']:
                    result = data['documents'][0]
                    lat, lng = float(result['y']), float(result['x'])
                    
                    # 서울시 범위 검증 (대략적)
                    if 37.4 <= lat <= 37.7 and 126.8 <= lng <= 127.2:
                        return {'lat': lat, 'lng': lng, 'status': 'API성공'}
                    else:
                        return {'lat': lat, 'lng': lng, 'status': 'API성공(범위외)'}
                else:
                    return {'lat': None, 'lng': None, 'status': 'API결과없음'}
            elif response.status_code == 429:
                print("   ⚠️ API 호출 한도 초과 - 잠시 대기...")
                time.sleep(5)
                return self.get_coordinates(address)  # 재시도
            else:
                return {'lat': None, 'lng': None, 'status': f'API오류({response.status_code})'}
                
        except Exception as e:
            return {'lat': None, 'lng': None, 'status': f'예외({str(e)[:20]})'}
    
    def get_district_center(self, address):
        """구별 중심 좌표 반환 (API 실패 시 사용)"""
        for district, coords in self.district_coords.items():
            if district in address:
                lat, lng = coords
                # 구 내에서 약간의 랜덤 변화 (±1km 정도)
                lat += np.random.uniform(-0.01, 0.01)
                lng += np.random.uniform(-0.01, 0.01)
                return {'lat': lat, 'lng': lng, 'status': f'구중심({district})'}
        
        # 서울시 중심 (알 수 없는 경우)
        return {'lat': 37.5665, 'lng': 126.9780, 'status': '서울중심'}

def convert_addresses_to_coordinates(df, batch_size=30, sample_size=None):
    """
    지오코딩 실행 (API + 백업 좌표)
    
    Args:
        df (pd.DataFrame): 주소가 포함된 데이터프레임
        batch_size (int): 배치 크기 (API 안정성을 위해 줄임)
        sample_size (int): 테스트용 샘플 크기 (None이면 전체 처리)
    
    Returns:
        pd.DataFrame: 좌표가 추가된 데이터프레임
    """
    df_coord = df.copy()
    geocoder = SimpleKakaoGeocoder()
    
    # 샘플링 (테스트용)
    if sample_size and sample_size < len(df_coord):
        print(f"🧪 테스트용 샘플링: {sample_size}건만 처리")
        df_coord = df_coord.head(sample_size)
    
    print(f"\n🗺️ 지오코딩 시작... (총 {len(df_coord):,}건)")
    
    if not geocoder.api_key:
        print("⚠️  API 키 없음 → 구별 중심 좌표 사용")
        return add_district_coordinates(df_coord, geocoder)
    
    print(f"✅ API 사용 가능 → 실제 지오코딩 진행")
    print(f"📍 배치 크기: {batch_size}개씩 처리")
    
    # 배치별 API 호출
    all_results = []
    failed_count = 0
    
    total_batches = (len(df_coord) + batch_size - 1) // batch_size
    
    for i in range(0, len(df_coord), batch_size):
        batch_end = min(i + batch_size, len(df_coord))
        batch = df_coord.iloc[i:batch_end]
        batch_num = i // batch_size + 1
        
        print(f"   배치 {batch_num}/{total_batches}: {i+1}-{batch_end} ({len(batch)}개)")
        
        for idx, row in batch.iterrows():
            # API 시도
            result = geocoder.get_coordinates(row['지오코딩주소'])
            
            if result and result['lat'] is not None:
                all_results.append({
                    'index': idx,
                    'lat': result['lat'],
                    'lng': result['lng'],
                    'status': result['status']
                })
            else:
                # API 실패 시 구별 중심 좌표 사용
                failed_count += 1
                backup = geocoder.get_district_center(row['지오코딩주소'])
                all_results.append({
                    'index': idx,
                    'lat': backup['lat'],
                    'lng': backup['lng'],
                    'status': backup['status']
                })
        
        # 배치 간 대기 (API 안정성)
        if batch_num < total_batches:
            time.sleep(1)
    
    # 결과 병합
    coord_df = pd.DataFrame(all_results)
    df_result = df_coord.merge(coord_df, left_index=True, right_on='index', how='left')
    df_result.drop('index', axis=1, inplace=True)
    df_result.rename(columns={'lat': '위도', 'lng': '경도', 'status': '좌표상태'}, inplace=True)
    
    # 결과 요약
    print("✅ 지오코딩 완료:")
    status_counts = df_result['좌표상태'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / len(df_result) * 100) if len(df_result) > 0 else 0
        print(f"   - {status}: {count:,}건 ({percentage:.1f}%)")
    
    # 좌표 범위 확인
    valid_coords = df_result['위도'].notna().sum()
    if valid_coords > 0:
        lat_range = f"{df_result['위도'].min():.4f} ~ {df_result['위도'].max():.4f}"
        lng_range = f"{df_result['경도'].min():.4f} ~ {df_result['경도'].max():.4f}"
        print(f"   🗺️ 좌표 범위: 위도 {lat_range}, 경도 {lng_range}")
    
    return df_result

def add_district_coordinates(df, geocoder):
    """구별 중심 좌표만 사용하는 백업 방법"""
    
    coordinates = []
    for idx, row in df.iterrows():
        coord = geocoder.get_district_center(row['지오코딩주소'])
        coordinates.append({
            'index': idx,
            'lat': coord['lat'],
            'lng': coord['lng'],
            'status': coord['status']
        })
    
    coord_df = pd.DataFrame(coordinates)
    df_result = df.merge(coord_df, left_index=True, right_on='index', how='left')
    df_result.drop('index', axis=1, inplace=True)
    df_result.rename(columns={'lat': '위도', 'lng': '경도', 'status': '좌표상태'}, inplace=True)
    
    print(f"✅ 구별 중심 좌표 할당 완료: {len(df_result)}건")
    return df_result

# 지오코딩 실행
if 'df_simplified' in locals() and df_simplified is not None:
    # 테스트를 위해 처음 100건만 처리 (전체 처리하려면 sample_size=None)
    print("\n💡 테스트를 위해 처음 100건만 처리합니다.")
    print("   전체 처리하려면 아래 코드에서 sample_size=None으로 변경하세요.")
    
    df_with_coords = convert_addresses_to_coordinates(df_simplified, batch_size=50, sample_size=None)
    
    if df_with_coords is not None:
        print(f"\n📍 좌표 변환 결과 미리보기:")
        display_cols = ['관리코드', '지오코딩주소', '공사상태', '위도', '경도', '좌표상태']
        print(df_with_coords[display_cols].head(5))


💡 테스트를 위해 처음 100건만 처리합니다.
   전체 처리하려면 아래 코드에서 sample_size=None으로 변경하세요.

🗺️ 지오코딩 시작... (총 533건)
✅ API 사용 가능 → 실제 지오코딩 진행
📍 배치 크기: 50개씩 처리
   배치 1/11: 1-50 (50개)


KeyboardInterrupt: 

In [ ]:
df_with_coords.head(10)

,관리코드,지오코딩주소,착공일,준공일,공사상태,위도,경도,좌표상태
0,일반-210728-75,서울시 서초구 잠원동 잠원동112 ~ 잠원동 잠원동112,2022-06-20,2022-08-31,완료,37.507930,127.003492,API성공
1,일반-210527-84,서울시 용산구 동자동 35-41 ~ 동자동 35-41부근,2022-06-16,2022-08-14,완료,37.523145,126.982056,구중심(용산구)
2,일반-210527-84,서울시 용산구 동자동 19-87 ~ 동자동 19-87부근,2022-06-16,2022-08-14,완료,37.542405,126.994573,구중심(용산구)
3,일반-210907-41,서울시 관악구 봉천동 958-14 ~ 봉천동 958-14,2022-06-15,2022-12-30,완료,37.484925,126.936693,API성공
4,인터넷-220604-01,서울시 종로구 이화동 149 ~ 이화동 149,2022-06-15,2022-06-19,완료,37.577306,127.003633,API성공
5,일반-210525-89,서울시 송파구 잠실동 183-4 ~ 잠실동 183-4,2022-06-13,2022-07-31,완료,37.511030,127.084156,API성공
6,일반-220527-35,서울시 종로구 홍지동 127-36 ~ 홍지동 127-36,2022-06-13,2022-07-30,완료,37.597624,126.959110,API성공
7,일반-210416-80,서울시 강남구 대치동 942-3 ~ 대치동 942-3,2022-06-02,2022-06-30,완료,37.505865,127.057580,API성공
8,일반-210416-20,서울시 관악구 봉천동 912-26 ~ 봉천동 912-26,2022-05-26,2022-12-30,완료,37.482305,126.945666,API성공
9,일반-210506-07,서울시 강남구 청담동 77-39 ~ 청담동 77-39,2022-05-26,2022-06-30,완료,37.522872,127.048431,API성공


In [ ]:
# =============================================================================
# 4. 최종 검증 및 저장
# =============================================================================

def validate_and_save_final_data(df, output_dir="../../data/processed"):
    """
    최종 데이터 검증 및 저장
    
    Args:
        df (pd.DataFrame): 최종 처리된 데이터프레임
        output_dir (str): 저장할 디렉터리
    
    Returns:
        dict: 검증 결과 요약
    """
    print(f"\n🔍 최종 데이터 검증...")
    
    # 1. 기본 통계
    total_records = len(df)
    valid_coords = (df['위도'].notna() & df['경도'].notna()).sum()
    valid_dates = (df['착공일'].notna() & df['준공일'].notna()).sum()
    ongoing_projects = (df['공사상태'] == '진행중').sum()
    
    summary = {
        'total_records': total_records,
        'valid_coordinates': valid_coords,
        'valid_dates': valid_dates,
        'ongoing_projects': ongoing_projects,
        'coord_success_rate': (valid_coords / total_records * 100) if total_records > 0 else 0,
        'date_success_rate': (valid_dates / total_records * 100) if total_records > 0 else 0
    }
    
    # 2. 좌표 범위 검증 (서울시 범위)
    if valid_coords > 0:
        seoul_coords = (
            (df['위도'] >= 37.4) & (df['위도'] <= 37.7) &
            (df['경도'] >= 126.8) & (df['경도'] <= 127.2)
        ).sum()
        summary['seoul_range_coords'] = seoul_coords
        summary['seoul_range_rate'] = (seoul_coords / valid_coords * 100) if valid_coords > 0 else 0
    
    # 3. 결과 출력
    print("=" * 60)
    print("📊 최종 데이터 요약")
    print("=" * 60)
    print(f"🎯 총 데이터: {summary['total_records']:,}건")
    print(f"📍 유효 좌표: {summary['valid_coordinates']:,}건 ({summary['coord_success_rate']:.1f}%)")
    print(f"📅 유효 날짜: {summary['valid_dates']:,}건 ({summary['date_success_rate']:.1f}%)")
    print(f"🚧 진행중 공사: {summary['ongoing_projects']:,}건")
    
    if 'seoul_range_coords' in summary:
        print(f"🗺️ 서울시 범위: {summary['seoul_range_coords']:,}건 ({summary['seoul_range_rate']:.1f}%)")
    
    print(f"\n🏢 공사 상태별 분포:")
    status_counts = df['공사상태'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / total_records * 100) if total_records > 0 else 0
        print(f"   - {status}: {count:,}건 ({percentage:.1f}%)")
    
    if '좌표상태' in df.columns:
        print(f"\n📍 좌표 획득 방법별 분포:")
        coord_status_counts = df['좌표상태'].value_counts()
        for status, count in coord_status_counts.items():
            percentage = (count / total_records * 100) if total_records > 0 else 0
            print(f"   - {status}: {count:,}건 ({percentage:.1f}%)")
    
    # 4. 데이터 저장
    try:
        os.makedirs(output_dir, exist_ok=True)
        
        # CSV 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_filename = f"road_excavation_processed_{timestamp}.csv"
        csv_path = os.path.join(output_dir, csv_filename)
        
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        file_size_mb = os.path.getsize(csv_path) / (1024 * 1024)
        
        print(f"\n💾 데이터 저장 완료:")
        print(f"   - 파일: {csv_path}")
        print(f"   - 크기: {file_size_mb:.2f} MB")
        print(f"   - 컬럼: {list(df.columns)}")
        
        summary['output_file'] = csv_path
        summary['file_size_mb'] = file_size_mb
        
        # 최신 파일 링크 생성
        latest_path = os.path.join(output_dir, "road_excavation_latest.csv")
        df.to_csv(latest_path, index=False, encoding='utf-8-sig')
        print(f"   - 최신버전: {latest_path}")
        
    except Exception as e:
        print(f"❌ 저장 실패: {str(e)}")
        summary['save_error'] = str(e)
    
    return summary

# 최종 검증 및 저장
if 'df_with_coords' in locals() and df_with_coords is not None:
    final_summary = validate_and_save_final_data(df_with_coords)
    
    print(f"\n🎉 FR-005 데이터 전처리 완료!")
    print("=" * 70)
    print("📋 최종 데이터셋 정보:")
    print(f"   - 총 레코드: {final_summary['total_records']:,}건")
    print(f"   - 좌표 성공률: {final_summary['coord_success_rate']:.1f}%")
    print(f"   - 날짜 성공률: {final_summary['date_success_rate']:.1f}%")
    print(f"   - 현재 진행중 공사: {final_summary['ongoing_projects']:,}건")
    
    if 'output_file' in final_summary:
        print(f"   - 저장 파일: {final_summary['output_file']}")
        print(f"   - 파일 크기: {final_summary['file_size_mb']:.2f} MB")
    
    print("\n🚀 다음 단계:")
    print("   1. 음성 안내 메시지 생성 로직 개발")
    print("   2. 사용자 위치 기반 공사 현장 필터링")
    print("   3. Azure Speech Services 연동")
    print("   4. 실시간 경로 안내 시스템 구축")

else:
    print("❌ 전처리 과정에서 오류가 발생했습니다.")
    print("💡 위의 단계들을 다시 확인해보세요.")
    print("\n🔍 문제 해결 방법:")
    print("   1. CSV 파일 경로 확인")
    print("   2. 필수 컬럼 존재 여부 확인")
    print("   3. 카카오 API 키 설정 확인")
    print("   4. 인터넷 연결 상태 확인")


🔍 최종 데이터 검증...
📊 최종 데이터 요약
🎯 총 데이터: 533건
📍 유효 좌표: 533건 (100.0%)
📅 유효 날짜: 372건 (69.8%)
🚧 진행중 공사: 34건
🗺️ 서울시 범위: 533건 (100.0%)

🏢 공사 상태별 분포:
   - 완료: 338건 (63.4%)
   - 불명: 161건 (30.2%)
   - 진행중: 34건 (6.4%)

📍 좌표 획득 방법별 분포:
   - API성공: 433건 (81.2%)
   - 구중심(강남구): 45건 (8.4%)
   - 구중심(강북구): 14건 (2.6%)
   - 구중심(성북구): 12건 (2.3%)
   - 구중심(서초구): 5건 (0.9%)
   - 구중심(동대문구): 5건 (0.9%)
   - 구중심(용산구): 4건 (0.8%)
   - 구중심(종로구): 3건 (0.6%)
   - 구중심(서대문구): 3건 (0.6%)
   - 구중심(관악구): 2건 (0.4%)
   - 구중심(구로구): 2건 (0.4%)
   - 구중심(강동구): 1건 (0.2%)
   - 구중심(노원구): 1건 (0.2%)
   - 구중심(마포구): 1건 (0.2%)
   - 구중심(은평구): 1건 (0.2%)
   - 구중심(송파구): 1건 (0.2%)

💾 데이터 저장 완료:
   - 파일: ../../data/processed\road_excavation_processed_20250623_143840.csv
   - 크기: 0.08 MB
   - 컬럼: ['관리코드', '지오코딩주소', '착공일', '준공일', '공사상태', '위도', '경도', '좌표상태']
   - 최신버전: ../../data/processed\road_excavation_latest.csv

🎉 FR-005 데이터 전처리 완료!
📋 최종 데이터셋 정보:
   - 총 레코드: 533건
   - 좌표 성공률: 100.0%
   - 날짜 성공률: 69.8%
   - 현재 진행중 공사: 34건
   - 저장 파일: ../../data/pr

In [ ]:
# =============================================================================
# 5. 추가 분석 및 시각화 (선택사항)
# =============================================================================

def analyze_construction_patterns(df):
    """공사 패턴 분석 (선택적 실행)"""
    
    if df is None or len(df) == 0:
        print("❌ 분석할 데이터가 없습니다.")
        return
    
    print(f"\n📊 공사 패턴 분석...")
    
    # 1. 월별 공사 시작 패턴
    if '착공일' in df.columns and df['착공일'].notna().sum() > 0:
        df['착공월'] = df['착공일'].dt.to_period('M')
        monthly_starts = df.groupby('착공월').size()
        
        print(f"📅 월별 착공 건수 (최근 12개월):")
        for month, count in monthly_starts.tail(12).items():
            print(f"   - {month}: {count:,}건")
    
    # 2. 공사 기간 분석
    if '착공일' in df.columns and '준공일' in df.columns:
        df['공사기간_일'] = (df['준공일'] - df['착공일']).dt.days
        valid_duration = df['공사기간_일'].dropna()
        
        if len(valid_duration) > 0:
            print(f"\n⏱️ 공사 기간 통계:")
            print(f"   - 평균: {valid_duration.mean():.0f}일")
            print(f"   - 중간값: {valid_duration.median():.0f}일")
            print(f"   - 최단: {valid_duration.min():.0f}일")
            print(f"   - 최장: {valid_duration.max():.0f}일")
    
    # 3. 지역별 공사 밀도
    if '좌표상태' in df.columns:
        coord_success = df.groupby('좌표상태').size()
        print(f"\n🗺️ 좌표 획득 현황:")
        for status, count in coord_success.items():
            print(f"   - {status}: {count:,}건")
    
    # 4. 현재 진행중인 공사 정보
    ongoing = df[df['공사상태'] == '진행중']
    if len(ongoing) > 0:
        print(f"\n🚧 현재 진행중인 공사 ({len(ongoing):,}건):")
        
        # 완료 예정일 기준 정렬
        if '준공일' in ongoing.columns:
            upcoming = ongoing.dropna(subset=['준공일']).sort_values('준공일')
            print(f"   📅 곧 완료 예정 (상위 5개):")
            for i, (_, row) in enumerate(upcoming.head(5).iterrows(), 1):
                days_left = (row['준공일'] - datetime.now()).days
                print(f"      {i}. {row.get('지오코딩주소', 'N/A')[:30]}... ({days_left}일 남음)")
    
    return df

# 패턴 분석 실행 (선택사항)
if 'df_with_coords' in locals() and df_with_coords is not None:
    print("\n" + "=" * 70)
    print("📈 추가 분석 실행 (선택사항)")
    print("=" * 70)
    
    try:
        df_analyzed = analyze_construction_patterns(df_with_coords)
    except Exception as e:
        print(f"⚠️ 분석 중 오류 발생: {str(e)}")

print("\n" + "=" * 70)
print("🏁 FR-005.ipynb 전체 실행 완료")
print("=" * 70)
print("💡 성공적으로 처리되었다면 다음 파일들이 생성됩니다:")
print("   - data/processed/road_excavation_processed_YYYYMMDD_HHMMSS.csv")
print("   - data/processed/road_excavation_latest.csv")


📈 추가 분석 실행 (선택사항)

📊 공사 패턴 분석...
📅 월별 착공 건수 (최근 12개월):
   - 2021-01: 1건
   - 2021-07: 7건
   - 2021-08: 16건
   - 2021-09: 8건
   - 2021-11: 1건
   - 2021-12: 16건
   - 2022-01: 17건
   - 2022-02: 2건
   - 2022-03: 20건
   - 2022-04: 119건
   - 2022-05: 131건
   - 2022-06: 8건

⏱️ 공사 기간 통계:
   - 평균: 406일
   - 중간값: 190일
   - 최단: 4일
   - 최장: 1845일

🗺️ 좌표 획득 현황:
   - API성공: 433건
   - 구중심(강남구): 45건
   - 구중심(강동구): 1건
   - 구중심(강북구): 14건
   - 구중심(관악구): 2건
   - 구중심(구로구): 2건
   - 구중심(노원구): 1건
   - 구중심(동대문구): 5건
   - 구중심(마포구): 1건
   - 구중심(서대문구): 3건
   - 구중심(서초구): 5건
   - 구중심(성북구): 12건
   - 구중심(송파구): 1건
   - 구중심(용산구): 4건
   - 구중심(은평구): 1건
   - 구중심(종로구): 3건

🚧 현재 진행중인 공사 (34건):
   📅 곧 완료 예정 (상위 5개):
      1. 서울시 강남구 청담동   77도 ~ 청담동   77-4... (6일 남음)
      2. 서울시 노원구 중계동   한글비석로262 ~ 상계동  ... (105일 남음)
      3. 서울시 성북구 하월곡동   230-2 ~ 하월곡동   ... (190일 남음)
      4. 서울시 성북구 하월곡동   오패산로 1가길 17-1, ... (190일 남음)
      5. 서울시 성북구 종암동   종암로 127-1, 3-173... (190일 남음)

🏁 FR-005.ipynb 전체 실행 완료
💡 성공적으로 처리되었다면 다음 파일들이 생성